
# executive_dashboard

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================================
# Paths
# ==========================================================

GOLD = Path("../data/gold")
FACT = GOLD / "facts"
DIM = GOLD / "dimensions"
MART = GOLD / "data_marts"

MART.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Tables
# ==========================================================

quote = pd.read_csv(FACT / "fact_quote.csv")
claim = pd.read_csv(FACT / "fact_claim.csv")
uw = pd.read_csv(FACT / "fact_underwriting.csv")
journey = pd.read_csv(FACT / "fact_customer_journey.csv")
ai = pd.read_csv(FACT / "fact_ai_interaction.csv")
date = pd.read_csv(DIM / "dim_date.csv")

# ==========================================================
# Scope dim_date to dates that actually appear in the data
# ==========================================================

dates_present = pd.concat([
    quote["date_sk"],
    claim["date_sk"],
    uw["underwriting_date_sk"],
    journey["date_sk"],
    ai["date_sk"]
]).dropna().unique()

date = date[date["date_sk"].isin(dates_present)]

# ==========================================================
# Quote Metrics
# ==========================================================

quote_summary = (
    quote
    .groupby("date_sk")
    .agg(
        quote_volume=("quote_sk", "count"),
        policies_issued=("conversion_flag", "sum"),   # confirm this is the right field — see note above
        premium_collected=("quoted_premium", "sum")
    )
    .reset_index()
)

quote_summary["conversion_rate"] = (
    quote_summary["policies_issued"] / quote_summary["quote_volume"]
) * 100

# ==========================================================
# Claim Metrics — approved/paid claims only
# ==========================================================

claim_summary = (
    claim[claim["claim_approved"] == True]
    .groupby("date_sk")
    .agg(
        claims_paid=("claim_flag", "sum"),
        claim_amount=("claim_amount", "sum")
    )
    .reset_index()
)

# ==========================================================
# Underwriting Metrics
# ==========================================================

uw_summary = (
    uw
    .groupby("underwriting_date_sk")
    .agg(
        manual_reviews=("manual_review_flag", "sum"),
        total_underwriting=("underwriting_sk", "count")
    )
    .reset_index()
    .rename(columns={"underwriting_date_sk": "date_sk"})
)

uw_summary["manual_underwriting_pct"] = (
    uw_summary["manual_reviews"] / uw_summary["total_underwriting"]
) * 100

# ==========================================================
# AI Metrics
# ==========================================================

ai_summary = (
    ai
    .groupby("date_sk")
    .agg(
        ai_usage=("interaction_sk", "count"),
        customer_satisfaction=("customer_rating", "mean")
    )
    .reset_index()
)

# ==========================================================
# Journey Metrics — sum per session first, then average
# per-session totals by date (fixes the stage-vs-journey
# grain mismatch)
# ==========================================================

session_totals = (
    journey
    .groupby("session_id")
    .agg(
        session_date_sk=("date_sk", "min"),
        total_duration=("duration_seconds", "sum")
    )
    .reset_index()
)

journey_summary = (
    session_totals
    .groupby("session_date_sk")
    .agg(average_journey_time=("total_duration", "mean"))
    .reset_index()
    .rename(columns={"session_date_sk": "date_sk"})
)

# ==========================================================
# Merge All KPIs
# ==========================================================

dashboard = (
    date
    .merge(quote_summary, on="date_sk", how="left")
    .merge(claim_summary, on="date_sk", how="left")
    .merge(uw_summary, on="date_sk", how="left")
    .merge(ai_summary, on="date_sk", how="left")
    .merge(journey_summary, on="date_sk", how="left")
)

# Only zero-fill count-type columns; leave rates/averages as
# NaN so Power BI shows a gap instead of a fabricated 0
count_cols = ["quote_volume", "policies_issued", "premium_collected",
              "claims_paid", "claim_amount", "ai_usage"]
dashboard[count_cols] = dashboard[count_cols].fillna(0)

# ==========================================================
# Claim Ratio
# ==========================================================

dashboard["claim_ratio"] = np.where(
    dashboard["premium_collected"] > 0,
    (dashboard["claim_amount"] / dashboard["premium_collected"]) * 100,
    np.nan
)

# ==========================================================
# Final Dashboard
# ==========================================================

executive_dashboard = dashboard[
    [
        "date_sk", "date", "year", "quarter", "month", "month_name", "financial_year",
        "quote_volume", "policies_issued", "conversion_rate",
        "premium_collected",
        "claims_paid", "claim_amount", "claim_ratio",
        "ai_usage",
        "manual_underwriting_pct",
        "customer_satisfaction",
        "average_journey_time"
    ]
]

# ==========================================================
# Save
# ==========================================================

executive_dashboard.to_csv(MART / "executive_dashboard.csv", index=False)
executive_dashboard.to_parquet(MART / "executive_dashboard.parquet", index=False)

print("=" * 70)
print("Executive Dashboard Created Successfully")
print("=" * 70)
print(executive_dashboard.head())
print(f"\nRows    : {len(executive_dashboard):,}")
print(f"Columns : {len(executive_dashboard.columns)}")

Executive Dashboard Created Successfully
    date_sk        date  year  quarter  month month_name financial_year  \
0  20250101  2025-01-01  2025        1      1    January      2024-2025   
1  20250102  2025-01-02  2025        1      1    January      2024-2025   
2  20250103  2025-01-03  2025        1      1    January      2024-2025   
3  20250104  2025-01-04  2025        1      1    January      2024-2025   
4  20250105  2025-01-05  2025        1      1    January      2024-2025   

   quote_volume  policies_issued  conversion_rate  premium_collected  \
0         282.0             25.0         8.865248          1367828.0   
1         268.0             23.0         8.582090          1296054.0   
2         256.0             21.0         8.203125          1241639.0   
3         276.0             28.0        10.144928          1334404.0   
4         285.0             17.0         5.964912          1394375.0   

   claims_paid  claim_amount  claim_ratio  ai_usage  manual_underwriting_pc

# underwriting_dashboard

In [2]:
import pandas as pd
from pathlib import Path

# ==========================================================
# Paths
# ==========================================================

GOLD = Path("../data/gold")

FACT = GOLD / "facts"
DIM = GOLD / "dimensions"
MART = GOLD / "data_marts"

MART.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Tables
# ==========================================================

uw = pd.read_csv(FACT / "fact_underwriting.csv")
policy = pd.read_csv(DIM / "dim_policy.csv")
vehicle = pd.read_csv(DIM / "dim_vehicle.csv")

# ==========================================================
# Merge Dimensions
# ==========================================================

dashboard = (
    uw
    .merge(
        policy[
            [
                "policy_sk",
                "policy_type",
                "coverage_type",
                "policy_status"
            ]
        ],
        on="policy_sk",
        how="left"
    )
    .merge(
        vehicle[
            [
                "vehicle_sk",
                "make",
                "model",
                "fuel_type",
                "segment",
                "vehicle_age_band"
            ]
        ],
        on="vehicle_sk",
        how="left"
    )
)

# ==========================================================
# Dashboard KPIs
# ==========================================================

dashboard["auto_approved_flag"] = (
    dashboard["underwriting_decision"] == "Approved"
).astype(int)

dashboard["manual_review_flag"] = (
    dashboard["manual_review_flag"]
).astype(int)

dashboard["rejected_flag"] = (
    dashboard["underwriting_decision"] == "Rejected"
).astype(int)

dashboard["stp_flag"] = (
    dashboard["processing_mode"] == "Straight Through Processing"
).astype(int)

dashboard["sla_breached_flag"] = (
    dashboard["sla_status"] == "SLA Breached"
).astype(int)

dashboard["high_risk_flag"] = (
    dashboard["risk_band"].isin(
        ["High", "Very High"]
    )
).astype(int)

dashboard["fraud_flag"] = (
    dashboard["fraud_probability"] >= 0.70
).astype(int)

# ==========================================================
# Final Dashboard Table
# ==========================================================

underwriting_dashboard = dashboard[
    [
        "underwriting_sk",
        "policy_sk",
        "customer_sk",
        "vehicle_sk",

        "policy_type",
        "coverage_type",
        "policy_status",

        "make",
        "model",
        "segment",
        "fuel_type",
        "vehicle_age_band",

        "risk_score",
        "risk_band",

        "fraud_probability",

        "underwriting_decision",

        "manual_review_flag",
        "auto_approved_flag",
        "rejected_flag",
        "stp_flag",

        "review_time_minutes",

        "premium_adjustment_pct",

        "rules_triggered",

        "ai_confidence",

        "processing_mode",

        "underwriter",

        "sla_status",
        "sla_breached_flag",

        "high_risk_flag",
        "fraud_flag",

        "recommendation",

        "model_version",

        "underwriting_date_sk"
    ]
]

# ==========================================================
# Save
# ==========================================================

underwriting_dashboard.to_csv(
    MART / "underwriting_dashboard.csv",
    index=False
)

underwriting_dashboard.to_parquet(
    MART / "underwriting_dashboard.parquet",
    index=False
)

print("=" * 70)
print("Underwriting Dashboard Mart Created Successfully")
print("=" * 70)

print(underwriting_dashboard.head())

print(f"\nRows    : {len(underwriting_dashboard):,}")
print(f"Columns : {len(underwriting_dashboard.columns)}")

Underwriting Dashboard Mart Created Successfully
   underwriting_sk  policy_sk  customer_sk  vehicle_sk    policy_type  \
0                1          1            1           1  Comprehensive   
1                2          2            2           2  Comprehensive   
2                3          3            3           3  Comprehensive   
3                4          4            4           4  Comprehensive   
4                5          5            5           5  Comprehensive   

  coverage_type policy_status  make model segment  ... ai_confidence  \
0        Silver        Active     1    M6      B2  ...         97.88   
1          Gold       Expired     1    M6      B2  ...         89.69   
2        Silver        Active     1    M6      B2  ...         87.76   
3          Gold        Active     1    M6      B2  ...         78.33   
4        Silver        Active     1    M1       A  ...         89.29   

               processing_mode  underwriter    sla_status  sla_breached_flag  \

# Sales Dashboard

In [7]:
import pandas as pd
from pathlib import Path

# ==========================================================
# Paths
# ==========================================================

GOLD = Path("../data/gold")

FACT = GOLD / "facts"
DIM = GOLD / "dimensions"
MART = GOLD / "data_marts"

MART.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Tables
# ==========================================================

quote = pd.read_csv(FACT / "fact_quote.csv")
customer = pd.read_csv(DIM / "dim_customer.csv")
channel = pd.read_csv(DIM / "dim_channel.csv")
date = pd.read_csv(DIM / "dim_date.csv")

# ==========================================================
# Merge Dimensions
# ==========================================================

dashboard = (
    quote
    .merge(customer, on="customer_sk", how="left")
    .merge(
        channel[["channel_sk", "channel_name"]],
        on="channel_sk", how="left"
    )
    .merge(
        date[["date_sk", "date", "year", "quarter", "month",
              "month_name", "week", "day_name"]],
        on="date_sk", how="left"
    )
)

# ==========================================================
# Sales KPIs
# ==========================================================

dashboard["quote_count"] = 1

# policy_issued_flag: a policy only exists if the quote actually
# converted, not just if the customer accepted the offer — same
# distinction fact_quote.conversion_flag was built to capture
dashboard["policy_issued_flag"] = dashboard["conversion_flag"]

dashboard["conversion_rate"] = dashboard["conversion_flag"] * 100

dashboard["mobile_quote_flag"] = (dashboard["device_type"] == "Mobile").astype(int)
dashboard["desktop_quote_flag"] = (dashboard["device_type"] == "Desktop").astype(int)

# rescaled to the real ~3,500-8,000 premium range — the original
# 8,000/15,000 thresholds assumed a much higher premium scale and
# left high_value_quote almost always 0
dashboard["high_value_quote"] = (dashboard["quoted_premium"] >= 7000).astype(int)
dashboard["medium_value_quote"] = (
    (dashboard["quoted_premium"] >= 5500) & (dashboard["quoted_premium"] < 7000)
).astype(int)
dashboard["low_value_quote"] = (dashboard["quoted_premium"] < 5500).astype(int)

# ==========================================================
# Final Dashboard
# ==========================================================

sales_dashboard = dashboard[
    [
        "quote_sk", "quote_id", "customer_sk", "channel_sk", "date_sk",
        "date", "year", "quarter", "month", "month_name", "week", "day_name",
        "channel_name", "gender", "customer_age", "region",
        "quote_source", "device_type", "quoted_premium", "quote_value_band",
        "days_since_last_contact", "quote_status", "quote_stage",
        "accepted_offer", "conversion_flag",
        "quote_count", "policy_issued_flag", "conversion_rate",
        "mobile_quote_flag", "desktop_quote_flag",
        "high_value_quote", "medium_value_quote", "low_value_quote"
    ]
]

# ==========================================================
# Save
# ==========================================================

sales_dashboard.to_csv(MART / "sales_dashboard.csv", index=False)
sales_dashboard.to_parquet(MART / "sales_dashboard.parquet", index=False)

print("=" * 70)
print("Sales Dashboard Mart Created Successfully")
print("=" * 70)
print(sales_dashboard.head())
print(f"\nRows    : {len(sales_dashboard):,}")
print(f"Columns : {len(sales_dashboard.columns)}")

Sales Dashboard Mart Created Successfully
   quote_sk quote_id  customer_sk  channel_sk   date_sk        date  year  \
0         1  Q100000            1         154  20250517  2025-05-17  2025   
1         2  Q100001            2         154  20250924  2025-09-24  2025   
2         3  Q100002            3         154  20250707  2025-07-07  2025   
3         4  Q100003            4         155  20251024  2025-10-24  2025   
4         5  Q100004            5         154  20250106  2025-01-06  2025   

   quarter  month month_name  ...  accepted_offer conversion_flag quote_count  \
0        2      5        May  ...               0               0           1   
1        3      9  September  ...               0               0           1   
2        3      7       July  ...               0               0           1   
3        4     10    October  ...               0               0           1   
4        1      1    January  ...               0               0           1   

  policy

# Claims Dashboard

In [4]:
import pandas as pd
from pathlib import Path

GOLD = Path("../data/gold")
FACT = GOLD / "facts"
DIM = GOLD / "dimensions"
MART = GOLD / "data_marts"
MART.mkdir(parents=True, exist_ok=True)

claim = pd.read_csv(FACT / "fact_claim.csv")
policy = pd.read_csv(DIM / "dim_policy.csv")
vehicle = pd.read_csv(DIM / "dim_vehicle.csv")
date = pd.read_csv(DIM / "dim_date.csv")

dashboard = (
    claim
    .merge(
        policy[[
            "policy_sk", "policy_id", "policy_type", "coverage_type",
            "policy_status", "policy_tenure", "policy_tenure_band"
        ]],
        on="policy_sk", how="left"
    )
    .merge(
        # vehicle_sk deliberately excluded here — claim already has its
        # own, real one from Silver; pulling a second copy via policy_id
        # collided with it and got silently renamed to vehicle_sk_x/_y
        vehicle[[
            "policy_id", "make", "model", "segment", "fuel_type",
            "vehicle_age_band", "premium_vehicle", "safety_rating"
        ]],
        on="policy_id", how="left"
    )
    .merge(
        date[[
            "date_sk", "date", "year", "quarter", "quarter_name",
            "month", "month_name", "week", "financial_year"
        ]],
        on="date_sk", how="left"
    )
)

dashboard["claim_count"] = 1
dashboard["paid_claim"] = dashboard["claim_approved"]
dashboard["high_fraud"] = (dashboard["fraud_risk"] == "High").astype(int)
dashboard["high_severity"] = (dashboard["claim_severity"] == "High").astype(int)
dashboard["fast_settlement"] = (dashboard["settlement_band"] == "Fast").astype(int)
dashboard["delayed_settlement"] = (dashboard["settlement_band"] == "Delayed").astype(int)

claims_dashboard = dashboard[
    [
        "claim_sk", "claim_id", "policy_sk", "vehicle_sk", "date_sk",
        "date", "year", "quarter", "month", "month_name", "week", "financial_year",
        "policy_type", "coverage_type", "policy_status", "policy_tenure_band",
        "make", "model", "segment", "fuel_type", "vehicle_age_band",
        "premium_vehicle", "safety_rating",
        "claim_amount", "claim_amount_band", "claim_flag", "claim_approved",
        "claim_severity", "fraud_risk", "settlement_days", "settlement_band",
        "claim_count", "paid_claim", "high_fraud", "high_severity",
        "fast_settlement", "delayed_settlement"
    ]
]

claims_dashboard.to_csv(MART / "claims_dashboard.csv", index=False)
claims_dashboard.to_parquet(MART / "claims_dashboard.parquet", index=False)

print("=" * 70)
print("Claims Dashboard Created Successfully")
print("=" * 70)
print(claims_dashboard.head())
print(f"\nRows    : {len(claims_dashboard):,}")
print(f"Columns : {len(claims_dashboard.columns)}")

Claims Dashboard Created Successfully
   claim_sk   claim_id  policy_sk  vehicle_sk   date_sk        date  year  \
0         1  CLM100000          1           1  20250926  2025-09-26  2025   
1         2  CLM100001          2           2  20261231  2026-12-31  2026   
2         3  CLM100002          3           3  20260503  2026-05-03  2026   
3         4  CLM100003          4           4  20260309  2026-03-09  2026   
4         5  CLM100004          5           5  20250417  2025-04-17  2025   

   quarter  month month_name  ...  claim_severity fraud_risk settlement_days  \
0        3      9  September  ...             Low        Low               0   
1        4     12   December  ...             Low        Low               0   
2        2      5        May  ...             Low        Low               0   
3        1      3      March  ...             Low        Low               0   
4        2      4      April  ...             Low        Low               0   

  settlement_band 

# customer360

In [5]:
import pandas as pd
from pathlib import Path

GOLD = Path("../data/gold")
FACT = GOLD / "facts"
DIM = GOLD / "dimensions"
MART = GOLD / "data_marts"
MART.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Tables
# ==========================================================

customer = pd.read_csv(DIM / "dim_customer.csv")
quote = pd.read_csv(FACT / "fact_quote.csv")
channel = pd.read_csv(DIM / "dim_channel.csv")
date = pd.read_csv(DIM / "dim_date.csv")
journey = pd.read_csv(FACT / "fact_customer_journey.csv")
ai = pd.read_csv(FACT / "fact_ai_interaction.csv")
underwriting = pd.read_csv(FACT / "fact_underwriting.csv")

# ==========================================================
# Aggregate stats — from the old customer_dashboard
# ==========================================================

journey_summary = (
    journey.groupby("customer_sk")
    .agg(
        total_journey_events=("journey_sk", "count"),
        avg_session_duration=("duration_seconds", "mean"),
        abandoned_sessions=("abandoned_flag", "sum"),
        ai_assistance_used=("ai_assistance_used", "max"),
    )
    .reset_index()
)

ai_summary = (
    ai.groupby("customer_sk")
    .agg(
        total_ai_interactions=("interaction_sk", "count"),
        avg_ai_confidence=("confidence_score", "mean"),
        avg_customer_rating=("customer_rating", "mean"),
        escalations=("escalation_required", "sum"),
        avg_response_time=("response_time_ms", "mean"),
    )
    .reset_index()
)

uw_summary = (
    underwriting.groupby("customer_sk")
    .agg(
        avg_risk_score=("risk_score", "mean"),
        manual_reviews=("manual_review_flag", "sum"),
        avg_review_time=("review_time_minutes", "mean"),
        fraud_probability=("fraud_probability", "max"),
    )
    .reset_index()
)

# ==========================================================
# Row-level context — from the old customer360
# ==========================================================

dashboard = (
    quote
    .merge(customer, on="customer_sk", how="left")
    .merge(channel[["channel_sk", "channel_name"]], on="channel_sk", how="left")
    .merge(
        date[["date_sk", "date", "year", "quarter", "quarter_name",
              "month", "month_name", "week", "financial_year"]],
        on="date_sk", how="left"
    )
    .merge(journey_summary, on="customer_sk", how="left")
    .merge(ai_summary, on="customer_sk", how="left")
    .merge(uw_summary, on="customer_sk", how="left")
)

# ==========================================================
# Fill missing values — only for the aggregate columns,
# which are genuinely 0/False when absent (unlike rates/
# averages elsewhere in the warehouse, these summarize
# count-type source facts)
# ==========================================================

dashboard.fillna(
    {
        "total_journey_events": 0, "avg_session_duration": 0, "abandoned_sessions": 0,
        "ai_assistance_used": False, "total_ai_interactions": 0, "avg_ai_confidence": 0,
        "avg_customer_rating": 0, "escalations": 0, "avg_response_time": 0,
        "avg_risk_score": 0, "manual_reviews": 0, "avg_review_time": 0, "fraud_probability": 0,
    },
    inplace=True,
)

# ==========================================================
# KPIs — conversion_flag throughout, not accepted_offer
# (the fix from two turns ago), value bins rescaled to match
# the real ~3,500-8,000 premium range
# ==========================================================

dashboard["customer_count"] = 1
dashboard["quote_count"] = 1
dashboard["converted_customer"] = dashboard["conversion_flag"]
dashboard["conversion_rate"] = dashboard["conversion_flag"] * 100

dashboard["high_value_quote"] = (dashboard["quoted_premium"] >= 7000).astype(int)
dashboard["mobile_customer"] = (dashboard["device_type"] == "Mobile").astype(int)
dashboard["desktop_customer"] = (dashboard["device_type"] == "Desktop").astype(int)
dashboard["returning_customer"] = (dashboard["previously_insured"] == 1).astype(int)
dashboard["new_customer"] = (dashboard["previously_insured"] == 0).astype(int)

dashboard["customer_segment"] = pd.cut(
    dashboard["quoted_premium"],
    bins=[0, 4500, 5500, 7000, float("inf")],
    labels=["Low Value", "Medium Value", "High Value", "Premium"],
)
dashboard["engagement_level"] = pd.cut(
    dashboard["total_ai_interactions"],
    bins=[-1, 1, 2, 3, 100],
    labels=["No Engagement", "Low", "Medium", "High"],
)
dashboard["risk_category"] = pd.cut(
    dashboard["avg_risk_score"],
    bins=[0, 30, 60, 80, 100],
    labels=["Low", "Medium", "High", "Very High"],
)

# ==========================================================
# Final Mart
# ==========================================================

customer360_dashboard = dashboard[
    [
        "customer_sk", "customer_id", "quote_sk", "quote_id",
        "date_sk", "date", "year", "quarter", "month", "month_name", "week", "financial_year",

        "gender", "customer_age", "age_band", "region",
        "customer_segment", "risk_profile", "risk_category",
        "has_driving_license", "previously_insured", "insurance_history",

        "channel_name", "device_type", "quote_source",
        "quote_status", "quote_stage", "quoted_premium",
        "accepted_offer", "conversion_flag",

        "total_journey_events", "avg_session_duration", "abandoned_sessions", "ai_assistance_used",
        "total_ai_interactions", "avg_ai_confidence", "avg_customer_rating", "escalations", "avg_response_time",
        "avg_risk_score", "manual_reviews", "avg_review_time", "fraud_probability",

        "customer_count", "quote_count", "converted_customer", "conversion_rate",
        "high_value_quote", "engagement_level",
        "mobile_customer", "desktop_customer", "returning_customer", "new_customer",
    ]
]

customer360_dashboard.to_csv(MART / "customer360_dashboard.csv", index=False)
customer360_dashboard.to_parquet(MART / "customer360_dashboard.parquet", index=False)

print("=" * 70)
print("Customer 360 Dashboard Created Successfully")
print("=" * 70)
print(customer360_dashboard.head())
print(f"\nRows    : {len(customer360_dashboard):,}")
print(f"Columns : {len(customer360_dashboard.columns)}")

Customer 360 Dashboard Created Successfully
   customer_sk customer_id  quote_sk quote_id   date_sk        date  year  \
0            1  CUST000001         1  Q100000  20250517  2025-05-17  2025   
1            2  CUST000002         2  Q100001  20250924  2025-09-24  2025   
2            3  CUST000003         3  Q100002  20250707  2025-07-07  2025   
3            4  CUST000004         4  Q100003  20251024  2025-10-24  2025   
4            5  CUST000005         5  Q100004  20250106  2025-01-06  2025   

   quarter  month month_name  ...  customer_count quote_count  \
0        2      5        May  ...               1           1   
1        3      9  September  ...               1           1   
2        3      7       July  ...               1           1   
3        4     10    October  ...               1           1   
4        1      1    January  ...               1           1   

  converted_customer  conversion_rate high_value_quote  engagement_level  \
0                  0      

# ai_monitoring_dashboard

In [6]:
import pandas as pd
from pathlib import Path

# ==========================================================
# Paths
# ==========================================================

GOLD = Path("../data/gold")

FACT = GOLD / "facts"
DIM = GOLD / "dimensions"
MART = GOLD / "data_marts"

MART.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Tables
# ==========================================================

ai = pd.read_csv(FACT / "fact_ai_interaction.csv")

customer = pd.read_csv(DIM / "dim_customer.csv")

date = pd.read_csv(DIM / "dim_date.csv")

# ==========================================================
# Merge Dimensions
# ==========================================================

dashboard = (

    ai

    .merge(

        customer,

        on="customer_sk",

        how="left"

    )

    .merge(

        date[
            [
                "date_sk",
                "date",
                "year",
                "quarter",
                "quarter_name",
                "month",
                "month_name",
                "week",
                "financial_year"
            ]
        ],

        on="date_sk",

        how="left"

    )

)

# ==========================================================
# AI KPIs
# ==========================================================

dashboard["interaction_count"] = 1

dashboard["resolved_interaction"] = (
    dashboard["resolved_flag"] == 1
).astype(int)

dashboard["escalated_interaction"] = (
    dashboard["escalation_required"] == 1
).astype(int)

dashboard["positive_sentiment"] = (
    dashboard["customer_sentiment"] == "Positive"
).astype(int)

dashboard["neutral_sentiment"] = (
    dashboard["customer_sentiment"] == "Neutral"
).astype(int)

dashboard["negative_sentiment"] = (
    dashboard["customer_sentiment"] == "Negative"
).astype(int)

dashboard["high_confidence"] = (
    dashboard["confidence_score"] >= 0.80
).astype(int)

dashboard["medium_confidence"] = (
    (dashboard["confidence_score"] >= 0.60) &
    (dashboard["confidence_score"] < 0.80)
).astype(int)

dashboard["low_confidence"] = (
    dashboard["confidence_score"] < 0.60
).astype(int)

dashboard["fast_response"] = (
    dashboard["response_speed"] == "Fast"
).astype(int)

dashboard["slow_response"] = (
    dashboard["response_speed"] == "Slow"
).astype(int)

dashboard["high_quality_response"] = (
    dashboard["ai_quality_score"] >= 90
).astype(int)

dashboard["avg_rating"] = dashboard["customer_rating"]

# ==========================================================
# Final Dashboard
# ==========================================================

ai_monitoring_dashboard = dashboard[
    [

        "interaction_sk",

        "customer_sk",

        "quote_sk",

        "date_sk",

        "date",

        "year",

        "quarter",

        "month",

        "month_name",

        "week",

        "financial_year",

        "session_id",

        "interaction_timestamp",

        "question_category",


        "customer_segment",

        "risk_profile",

        "region",

        "gender",

        "customer_age",

        "user_query",

        "ai_response",

        "response_time_ms",

        "response_speed",

        "confidence_score",

        "token_count",

        "customer_rating",

        "customer_sentiment",

        "resolved_flag",

        "escalation_required",

        "ai_model_version",

        "ai_quality_score",

        "interaction_count",

        "resolved_interaction",

        "escalated_interaction",

        "positive_sentiment",

        "neutral_sentiment",

        "negative_sentiment",

        "high_confidence",

        "medium_confidence",

        "low_confidence",

        "fast_response",

        "slow_response",

        "high_quality_response",

        "avg_rating"

    ]
]

# ==========================================================
# Save
# ==========================================================

ai_monitoring_dashboard.to_csv(
    MART / "ai_monitoring_dashboard.csv",
    index=False
)

ai_monitoring_dashboard.to_parquet(
    MART / "ai_monitoring_dashboard.parquet",
    index=False
)

print("=" * 70)
print("AI Monitoring Dashboard Created Successfully")
print("=" * 70)

print(ai_monitoring_dashboard.head())

print(f"\nRows    : {len(ai_monitoring_dashboard):,}")
print(f"Columns : {len(ai_monitoring_dashboard.columns)}")

AI Monitoring Dashboard Created Successfully
   interaction_sk  customer_sk  quote_sk   date_sk        date  year  quarter  \
0               1            1         1  20250517  2025-05-17  2025        2   
1               2            1         1  20250518  2025-05-18  2025        2   
2               3            1         1  20250517  2025-05-17  2025        2   
3               4            2         2  20250926  2025-09-26  2025        3   
4               5            2         2  20250925  2025-09-25  2025        3   

   month month_name  week  ... positive_sentiment neutral_sentiment  \
0      5        May    20  ...                  1                 0   
1      5        May    20  ...                  0                 0   
2      5        May    20  ...                  0                 1   
3      9  September    39  ...                  1                 0   
4      9  September    39  ...                  0                 0   

  negative_sentiment high_confidence medi